# SIRCH - Evaluation Colab

Evaluation du modele `sirch_model.h5` sauvegarde dans Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive monte.')

In [ ]:
import glob
import os
import time
import cv2
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

N_FRAMES = 20
IMG_SIZE = 224
MODEL_PATH = '/content/drive/MyDrive/SIRCH/models/sirch_model.h5'
DATASETS_DIR = '/content/drive/MyDrive/SIRCH/datasets'
SPORT_DIR = '/content/drive/MyDrive/SIRCH/test_sport'
DANCE_DIR = '/content/drive/MyDrive/SIRCH/test_danse'

model = tf.keras.models.load_model(MODEL_PATH)
print('Modele charge.')

In [ ]:
def load_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return np.zeros((1, N_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
    frame_indices = np.linspace(0, max(total - 1, 0), N_FRAMES).astype(int)
    frames = []
    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = cap.read()
        if not ok:
            frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frames.append(frame.astype(np.float32))
    cap.release()
    frames = np.stack(frames, axis=0)
    frames = tf.keras.applications.efficientnet.preprocess_input(frames)
    return np.expand_dims(frames, axis=0)

def collect_videos(folder):
    files = []
    for ext in ('*.mp4', '*.avi', '*.mov', '*.mkv'):
        files.extend(glob.glob(os.path.join(folder, '**', ext), recursive=True))
    return files

def predict_paths(paths):
    scores = []
    start = time.time()
    for path in paths:
        scores.append(float(model.predict(load_video_frames(path), verbose=0).ravel()[0]))
    elapsed = time.time() - start
    ms_per_frame = (elapsed / max(len(paths) * N_FRAMES, 1)) * 1000
    return np.array(scores), ms_per_frame

In [ ]:
violence_paths = collect_videos(os.path.join(DATASETS_DIR, 'RLVS', 'Violence')) + collect_videos(os.path.join(DATASETS_DIR, 'RWF-2000', 'val', 'Fight'))
non_violence_paths = collect_videos(os.path.join(DATASETS_DIR, 'RLVS', 'NonViolence')) + collect_videos(os.path.join(DATASETS_DIR, 'RWF-2000', 'val', 'NonFight'))
paths = violence_paths + non_violence_paths
truth = np.array([1] * len(violence_paths) + [0] * len(non_violence_paths))

scores, ms_per_frame = predict_paths(paths)
preds = (scores >= 0.5).astype(int)

print(f'Accuracy : {accuracy_score(truth, preds):.4f}')
print(f'Precision : {precision_score(truth, preds, zero_division=0):.4f}')
print(f'Rappel : {recall_score(truth, preds, zero_division=0):.4f}')
print(f'F1-score : {f1_score(truth, preds, zero_division=0):.4f}')
print(f'Temps inference moyen : {ms_per_frame:.2f} ms/frame')
print('Matrice de confusion :')
print(confusion_matrix(truth, preds))
print('Reference a battre : F1 = 86.39% (Abdullah et al. 2023 sur RLVS seul)')

In [ ]:
def false_positive_rate(folder):
    paths = collect_videos(folder)
    if not paths:
        print(f'Aucune video trouvee dans {folder}')
        return None
    scores, _ = predict_paths(paths)
    rate = float(np.mean(scores >= 0.5))
    print(f'Taux de faux positifs pour {folder} : {rate:.4f}')
    return rate

false_positive_rate(SPORT_DIR)
false_positive_rate(DANCE_DIR)